# 03 — ML · Predicting negative reviews

**Goal:** Predict at the moment an order is placed whether it will get a
**bad review (1–2 stars)**.

**Rule (avoids data leakage):** use only features that are known *at order time*.
- ✅ allowed: price, shipping cost, installments, estimated delivery time, customer state
- ❌ not allowed: actual delivery time, "was it late", the review text

**Known limitation:** we only use orders that were later marked as `delivered`.
That is a common simplification, although strictly this status is not known at order time.

## 1 · Build the feature table

One row per order. We load small tables from the database, each with one row per order,
and glue them together.

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

# find the database (works locally and on Kaggle)
_db = [Path("../data/raw/olist.sqlite"), Path("data/raw/olist.sqlite"),
       Path("/kaggle/input/brazilian-ecommerce/olist.sqlite")]
con = sqlite3.connect(next((p for p in _db if p.exists()), _db[0]))

def q(sql):
    return pd.read_sql_query(sql, con)

# --- one small table per topic, each with ONE row per order -------------------

# the orders themselves (only delivered ones, see limitation above)
orders = q("""
    SELECT order_id, customer_id, order_purchase_timestamp, order_estimated_delivery_date
    FROM orders
    WHERE order_status = 'delivered'
""")

# the review score per order (if an order has several reviews, take the worst one)
reviews = q("SELECT order_id, MIN(review_score) AS review_score FROM order_reviews GROUP BY order_id")

# the items per order: how many, total price, total shipping cost
items = q("""
    SELECT order_id, COUNT(*) AS n_items, SUM(price) AS price, SUM(freight_value) AS freight
    FROM order_items
    GROUP BY order_id
""")

# the payment per order: maximum number of installments
pay = q("SELECT order_id, MAX(payment_installments) AS installments FROM order_payments GROUP BY order_id")

# the customer's state
cust = q("SELECT customer_id, customer_state FROM customers")

# --- glue everything together (merge = JOIN in pandas) ------------------------
df = (orders.merge(reviews, on="order_id")      # add the review score
            .merge(items, on="order_id")        # add items / price / freight
            .merge(pay, on="order_id")          # add installments
            .merge(cust, on="customer_id"))     # add customer state

print(df.shape)

(95831, 10)


## 2 · Target and features

**Target** (what we want to predict): `bad_review` = 1 if the review has 1–2 stars, else 0.

**Features** (what the model may use): everything known at order time.
Besides the raw columns we create two new ones:
- `est_delivery_days` — the promised delivery time in days (the estimate is shown at purchase)
- `freight_ratio` — shipping cost relative to the price

In [2]:
# TARGET: 1 = bad review (1 or 2 stars), 0 = everything else
df["bad_review"] = (df["review_score"] <= 2).astype(int)

# FEATURE: promised delivery time in days (estimated date minus purchase date)
df["est_delivery_days"] = (pd.to_datetime(df["order_estimated_delivery_date"])
                           - pd.to_datetime(df["order_purchase_timestamp"])).dt.days

# FEATURE: shipping cost relative to the price (expensive shipping can annoy customers)
df["freight_ratio"] = df["freight"] / df["price"]

# quick check: how many rows and how many bad reviews?
print(df.shape, "| bad reviews:", round(100 * df["bad_review"].mean(), 1), "%")
df.head()

(95831, 13) | bad reviews: 12.8 %


,order_id,customer_id,order_purchase_timestamp,order_estimated_delivery_date,review_score,n_items,price,freight,installments,customer_state,bad_review,est_delivery_days,freight_ratio
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,2017-10-18 00:00:00,4,1,29.99,8.72,1,SP,0,15,0.290764
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,2018-08-13 00:00:00,4,1,118.70,22.76,1,BA,0,19,0.191744
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,2018-09-04 00:00:00,5,1,159.90,19.22,3,GO,0,26,0.120200
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18 19:28:06,2017-12-15 00:00:00,5,1,45.00,27.20,1,RN,0,26,0.604444
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 21:18:39,2018-02-26 00:00:00,5,1,19.90,8.72,1,SP,0,12,0.438191


**Read:** 95,831 orders, one row each. About **12.8 %** have a bad review, so the classes
are **imbalanced** — a model that always says "not bad" would already be right 87 % of the
time. That is why we will look at *precision* and *recall* later, not only accuracy.

## 3 · Train / test split

We split the data: the model **learns on 80 %** (train) and is **tested on 20 %** it has never
seen (test). `stratify=y` keeps the share of bad reviews (12.8 %) the same in both parts.

In [3]:
from sklearn.model_selection import train_test_split

# FEATURES the model may use (all known at order time)
num_features = ["price", "freight", "n_items", "installments", "est_delivery_days", "freight_ratio"]
cat_features = ["customer_state"]          # a category (text), needs special handling

X = df[num_features + cat_features]        # inputs
y = df["bad_review"]                       # what we want to predict

# 80 % train / 20 % test.  random_state=42 makes the split repeatable.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("train:", X_train.shape, "| test:", X_test.shape)
print("bad reviews  train:", round(100 * y_train.mean(), 1), "%  |  test:", round(100 * y_test.mean(), 1), "%")

train: (76664, 7) | test: (19167, 7)
bad reviews  train: 12.8 %  |  test: 12.8 %


## 4 · Baseline model: Logistic Regression

A simple first model. Two preparation steps are needed:
- **Scale** the number columns (`StandardScaler`) so that price in the hundreds and
  items in the single digits are comparable
- **One-hot encode** the state (`OneHotEncoder`): turns the text into 0/1 columns

`class_weight="balanced"` tells the model to take the rare bad reviews more seriously.
Without it, the model would almost always predict "not bad".

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# preparation: scale numbers, one-hot encode the state
prep = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
])

# pipeline = preparation + model in one object
baseline = Pipeline([
    ("prep", prep),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

baseline.fit(X_train, y_train)             # learn on the training data

pred = baseline.predict(X_test)            # predicted class: 0 or 1
proba = baseline.predict_proba(X_test)[:, 1]   # predicted probability of a bad review

### Evaluation

- **Confusion matrix:** rows = reality, columns = prediction
- **Precision (bad):** of all orders the model flagged as "bad", how many really were?
- **Recall (bad):** of all truly bad orders, how many did the model catch?
- **ROC-AUC:** how well the model separates bad from good (0.5 = coin flip, 1.0 = perfect)

In [5]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

# confusion matrix with readable labels
cm = pd.DataFrame(confusion_matrix(y_test, pred),
                  index=["really ok", "really bad"], columns=["predicted ok", "predicted bad"])
print(cm, "\n")

# precision / recall per class
print(classification_report(y_test, pred, target_names=["ok", "bad"], digits=3))

print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
print("Accuracy:", round(accuracy_score(y_test, pred), 3),
      "| always predicting 'ok' would give:", round(1 - y_test.mean(), 3))

            predicted ok  predicted bad
really ok          11669           5036
really bad          1278           1184 

              precision    recall  f1-score   support

          ok      0.901     0.699     0.787     16705
         bad      0.190     0.481     0.273      2462

    accuracy                          0.671     19167
   macro avg      0.546     0.590     0.530     19167
weighted avg      0.810     0.671     0.721     19167

ROC-AUC: 0.612
Accuracy: 0.671 | always predicting 'ok' would give: 0.872


In [6]:
# which numeric features push the risk up? (coefficients on scaled features)
coef = pd.Series(baseline.named_steps["model"].coef_[0][:len(num_features)], index=num_features)
coef.sort_values(key=abs, ascending=False).round(3)

n_items              0.307
installments         0.045
freight              0.036
price                0.035
est_delivery_days    0.027
freight_ratio        0.006
dtype: float64

**Read:**
- **ROC-AUC 0.61** — only a little better than a coin flip (0.5). With order-time features
  alone, the signal is weak. That makes sense: the strongest driver found in the SQL part
  (*late delivery*) is not known when the order is placed.
- **Recall for bad reviews 48 %, precision 19 %:** the model catches about half of the bad
  orders, but most of its warnings are false alarms. Accuracy (67 %) is even *below* the
  trivial "always ok" guess (87 %) — the price for making the model look for the rare class.
- **`n_items` is the strongest feature:** orders with more items have a higher risk of a
  bad review (hypothesis: items can arrive separately or one of them can be wrong).

**Next:** add more order-time features (distance customer ↔ seller, product category,
seller history), try a stronger model (Random Forest), and later compare with variant B
(including the actual delivery time) to show how much it adds.

## 5 · Random Forest (same features)

A **Random Forest** is many decision trees that vote together. Unlike Logistic Regression it
can find non-linear patterns (e.g. "risky only if many items *and* a long delivery").
Trees do not need scaled numbers, but the state still needs one-hot encoding.

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score

# preparation: numbers go through unchanged (trees don't need scaling), state is one-hot encoded
prep_rf = ColumnTransformer([
    ("num", "passthrough", num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
])

forest = Pipeline([
    ("prep", prep_rf),
    ("model", RandomForestClassifier(
        n_estimators=200,                    # number of trees
        max_depth=12,                        # limit tree depth (avoids memorising the data)
        min_samples_leaf=30,                 # every leaf needs at least 30 orders
        class_weight="balanced_subsample",   # take the rare bad reviews seriously
        n_jobs=-1,                           # use all CPU cores
        random_state=42)),
])
forest.fit(X_train, y_train)

# helper: the three numbers we care about, for one model
def scores(model):
    proba = model.predict_proba(X_test)[:, 1]
    pred = model.predict(X_test)
    return {"ROC-AUC": round(roc_auc_score(y_test, proba), 3),
            "recall (bad)": round(recall_score(y_test, pred), 2),
            "precision (bad)": round(precision_score(y_test, pred), 2)}

pd.DataFrame({"Logistic Regression": scores(baseline), "Random Forest": scores(forest)}).T

,ROC-AUC,recall (bad),precision (bad)
Logistic Regression,0.612,0.48,0.19
Random Forest,0.617,0.42,0.20


**Read:** The Random Forest is **hardly better** than the simple baseline (ROC-AUC about 0.61
for both). So the limit is not the model — it is the **information**: with only order-time
features, there is little signal to find. A stronger algorithm cannot fix that.

## 6 · Variant B: add the actual delivery

What if we also use what happens **after** the order: the *actual* delivery time and whether
it was late? This is not available at order time, but it is known on the **delivery day** —
still before the customer writes the review. We compare both variants on the same split.

In [8]:
# load the actual delivery date and build three new features
deliv = q("SELECT order_id, order_delivered_customer_date AS delivered FROM orders")
df_b = df.merge(deliv, on="order_id")
df_b = df_b.dropna(subset=["delivered"])          # a few orders have no delivery date

delivered = pd.to_datetime(df_b["delivered"])
df_b["actual_delivery_days"] = (delivered - pd.to_datetime(df_b["order_purchase_timestamp"])).dt.days
df_b["days_late"] = (delivered - pd.to_datetime(df_b["order_estimated_delivery_date"])).dt.days  # <0 = early
df_b["is_late"] = (df_b["days_late"] > 0).astype(int)

print("orders used:", len(df_b), "| removed (no delivery date):", len(df) - len(df_b))

orders used: 95823 | removed (no delivery date): 8


In [9]:
delivery_features = ["actual_delivery_days", "days_late", "is_late"]

# one common train/test split, so all four models are compared fairly
y_b = df_b["bad_review"]
idx_train, idx_test = train_test_split(df_b.index, test_size=0.2, random_state=42, stratify=y_b)

results = {}
for variant, nums in [("A: at order time", num_features),
                      ("B: with delivery", num_features + delivery_features)]:
    for name, clf in [
        ("Logistic Regression", LogisticRegression(max_iter=1000, class_weight="balanced")),
        ("Random Forest", RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=30,
                                                 class_weight="balanced_subsample", n_jobs=-1, random_state=42)),
    ]:
        pipe = Pipeline([
            ("prep", ColumnTransformer([("num", StandardScaler(), nums),
                                        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)])),
            ("model", clf),
        ])
        pipe.fit(df_b.loc[idx_train, nums + cat_features], y_b[idx_train])

        X_t = df_b.loc[idx_test, nums + cat_features]
        proba, pred = pipe.predict_proba(X_t)[:, 1], pipe.predict(X_t)
        results[(variant, name)] = {"ROC-AUC": round(roc_auc_score(y_b[idx_test], proba), 3),
                                    "recall (bad)": round(recall_score(y_b[idx_test], pred), 2),
                                    "precision (bad)": round(precision_score(y_b[idx_test], pred), 2)}
        if variant.startswith("B") and name == "Random Forest":
            forest_b, nums_b = pipe, nums           # keep this one to look at feature importance

pd.DataFrame(results).T

ROC-AUC  recall (bad)  precision (bad)
A: at order time Logistic Regression    0.610          0.48             0.19
                 Random Forest          0.616          0.43             0.20
B: with delivery Logistic Regression    0.740          0.52             0.37
                 Random Forest          0.743          0.53             0.39

In [10]:
# which features does the best model (B, Random Forest) rely on most?
importance = pd.Series(forest_b.named_steps["model"].feature_importances_[:len(nums_b)], index=nums_b)
importance.sort_values(ascending=False).round(3)

days_late               0.287
actual_delivery_days    0.200
is_late                 0.198
n_items                 0.124
freight                 0.069
price                   0.030
est_delivery_days       0.028
freight_ratio           0.027
installments            0.011
dtype: float64

**Read:**
- **Variant A (order time):** ROC-AUC about 0.61 — weak, for both models.
- **Variant B (with delivery):** ROC-AUC about **0.74**, and **precision roughly doubles**
  (from about 0.19 to about 0.38) at similar recall. Knowing the real delivery makes a big difference.
- **The top features are all about the delivery:** `days_late`, `actual_delivery_days`, `is_late`.
  This confirms the finding from the SQL part (late orders: 2.57 vs. 4.29 stars).
- **Random Forest ≈ Logistic Regression** in both variants — the signal is simple and mostly linear.

**What it means:** at order time, bad reviews are hard to predict; but on the **delivery day** the
model can flag risky orders. A practical use: proactively contact customers whose package arrived
late — *before* they write a bad review.